<a href="https://colab.research.google.com/github/misspresident-codes/decodelabs-_tasks02/blob/main/Project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import precision_score, recall_score, f1_score



In [ ]:
data = pd.read_csv("/content/fraudTrain.csv")


In [ ]:
data = data.sample(n=50000, random_state=42)

In [ ]:
data

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
1045211,1045211,2020-03-09 15:09:26,577588686219,fraud_Towne LLC,misc_pos,194.51,James,Strickland,M,25454 Leonard Lake,...,40.6153,-79.4545,972,Public relations account executive,1997-10-23,fff87d4340ef756a592eac652493cf6b,1362841766,40.420453,-78.865012,0
547406,547406,2019-08-22 15:49:01,30376238035123,fraud_Friesen Ltd,health_fitness,52.32,Cynthia,Davis,F,7177 Steven Forges,...,42.8250,-124.4409,217,Retail merchandiser,1928-10-01,d0ad335af432f35578eea01d639b3621,1345650541,42.758860,-123.636337,0
110142,110142,2019-03-04 01:34:16,4658490815480264,fraud_Mohr Inc,shopping_pos,6.53,Tara,Richards,F,4879 Cristina Station,...,39.9636,-79.7853,184,Systems developer,1945-11-04,87f26e3ea33f4ff4c7a8bad2c7f48686,1330824856,40.475159,-78.898190,0
1285953,1285953,2020-06-16 20:04:38,3514897282719543,fraud_Gaylord-Powlowski,home,7.33,Steven,Faulkner,M,841 Cheryl Centers Suite 115,...,42.9580,-77.3083,10717,Cytogeneticist,1952-10-13,9c34015321c0fa2ae6fd20f9359d1d3e,1371413078,43.767506,-76.542384,0
271705,271705,2019-05-14 05:54:48,6011381817520024,"fraud_Christiansen, Goyette and Schamberger",gas_transport,64.29,Kristen,Allen,F,8619 Lisa Manors Apt. 871,...,41.6423,-104.1974,635,Product/process development scientist,1973-07-13,198437c05676f485e9be04449c664475,1336974888,41.040392,-104.092324,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
480706,480706,2019-07-29 14:12:34,4926376199189801,fraud_Wilkinson LLC,personal_care,130.31,Claire,Davis,F,83685 Matthew Center Suite 870,...,36.3011,-91.5281,4726,Pharmacologist,1977-06-07,7ef2a4862bb06f688899bdc14f46591e,1343571154,35.709632,-92.208235,0
20742,20742,2019-01-13 14:47:18,2712209726293386,"fraud_Heathcote, Yost and Kertzmann",shopping_net,485.74,Jenna,Brooks,F,50872 Alex Plain Suite 088,...,30.4066,-91.1468,378909,"Designer, furniture",1977-02-22,a4009fb0aac31d310f41939a9f22e31a,1326466038,30.615228,-91.011859,0
51176,51176,2019-01-30 21:57:23,3589255887819806,fraud_Thompson-Gleason,health_fitness,131.19,David,Miller,M,622 Bradley Knoll Apt. 758,...,39.6991,-78.1762,3766,Press photographer,1984-02-14,b222bc4d36b406b29bc006023247113a,1327960643,38.757823,-78.689289,0
1216,1216,2019-01-01 14:25:35,4745996322265,fraud_Torp-Lemke,misc_pos,108.54,Carrie,Washington,F,6114 Adams Harbor Suite 096,...,41.4802,-86.6919,1423,"Psychologist, forensic",1998-10-07,ef3bf221feec64e27a18e2c8c52522c5,1325427935,42.307120,-87.621439,0


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 50000 entries, 1045211 to 1146033
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             50000 non-null  int64  
 1   trans_date_trans_time  50000 non-null  object 
 2   cc_num                 50000 non-null  int64  
 3   merchant               50000 non-null  object 
 4   category               50000 non-null  object 
 5   amt                    50000 non-null  float64
 6   first                  50000 non-null  object 
 7   last                   50000 non-null  object 
 8   gender                 50000 non-null  object 
 9   street                 50000 non-null  object 
 10  city                   50000 non-null  object 
 11  state                  50000 non-null  object 
 12  zip                    50000 non-null  int64  
 13  lat                    50000 non-null  float64
 14  long                   50000 non-null  float64
 15 

In [ ]:
data = data.drop([
    'Unnamed: 0',
    'cc_num',
    'first',
    'last',
    'street',
    'trans_num'
], axis=1)

In [ ]:
data['trans_date_trans_time'] = pd.to_datetime(data['trans_date_trans_time'], errors='coerce')
data['dob'] = pd.to_datetime(data['dob'], errors='coerce')

data["transaction_hour"] = data["trans_date_trans_time"].dt.hour
data["transaction_day"] = data["trans_date_trans_time"].dt.day
data["customer_age"] = (
    data["trans_date_trans_time"].dt.year -
    data["dob"].dt.year
)

In [ ]:
data = data.drop(['trans_date_trans_time', 'dob'], axis=1)


In [ ]:
categorical_columns = [
    'merchant',
    'category',
    'gender',
    'city',
    'state',
    'job'
]

In [ ]:
encoder = LabelEncoder()

In [ ]:
for column in categorical_columns:
    data[column] = encoder.fit_transform(
        data[column].astype(str)
    )

In [ ]:
data = data.dropna()


In [ ]:
X = data.drop("is_fraud", axis=1)
y = data["is_fraud"]

In [ ]:
print(data["is_fraud"].value_counts())

is_fraud
0    49705
1      295
Name: count, dtype: int64


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
logistic_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("model", LogisticRegression(max_iter=1000))
])

In [ ]:
logistic_pipeline.fit(X_train, y_train)


Pipeline(steps=[('scaler', StandardScaler()), ('smote', SMOTE(random_state=42)),
                ('model', LogisticRegression(max_iter=1000))])

In [ ]:
lr_predictions = logistic_pipeline.predict(X_test)
lr_probabilities = logistic_pipeline.predict_proba(X_test)[:, 1]

In [ ]:
print("========== LOGISTIC REGRESSION ==========")

========== LOGISTIC REGRESSION ==========


In [ ]:
print(classification_report(y_test, lr_predictions))

              precision    recall  f1-score   support

           0       1.00      0.94      0.97      9941
           1       0.06      0.66      0.11        59

    accuracy                           0.94     10000
   macro avg       0.53      0.80      0.54     10000
weighted avg       0.99      0.94      0.96     10000



In [ ]:
lr_auc = roc_auc_score(
    y_test,
    lr_probabilities
)

In [ ]:
print("ROC-AUC Score:", lr_auc)

ROC-AUC Score: 0.8056107304281703


In [ ]:
rf_pipeline = Pipeline([
    ("smote", SMOTE(random_state=42)),
    ("model", RandomForestClassifier(random_state=42))
])

In [ ]:
random_search = RandomizedSearchCV(
    rf_pipeline,
    param_distributions={
        "model__n_estimators": [50, 100, 150],
        "model__max_depth": [5, 10, 15, None]
    },
    n_iter=4,
    cv=2,
    scoring="recall",
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=2,
                   estimator=Pipeline(steps=[('smote', SMOTE(random_state=42)),
                                             ('model',
                                              RandomForestClassifier(random_state=42))]),
                   n_iter=4, n_jobs=-1,
                   param_distributions={'model__max_depth': [5, 10, 15, None],
                                        'model__n_estimators': [50, 100, 150]},
                   random_state=42, scoring='recall')

In [ ]:
best_rf_model = random_search.best_estimator_
rf_predictions = best_rf_model.predict(X_test)
rf_probabilities = best_rf_model.predict_proba(X_test)[:, 1]
print("\n========== RANDOM FOREST ==========")
print(classification_report(y_test, rf_predictions))
rf_auc = roc_auc_score(y_test, rf_probabilities)
print("ROC-AUC Score:", rf_auc)
print("\n========== BEST PARAMETERS ==========")
print(random_search.best_params_)


========== RANDOM FOREST ==========
              precision    recall  f1-score   support

           0       1.00      0.96      0.98      9941
           1       0.10      0.69      0.17        59

    accuracy                           0.96     10000
   macro avg       0.55      0.83      0.58     10000
weighted avg       0.99      0.96      0.97     10000

ROC-AUC Score: 0.8813832117970603

========== BEST PARAMETERS ==========
{'model__n_estimators': 50, 'model__max_depth': 5}


In [ ]:
print("\n========== MODEL COMPARISON ==========")
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "ROC_AUC": [lr_auc, rf_auc]
})
print(comparison)
if rf_auc > lr_auc:
    print("\nRandom Forest performed better on this dataset.")
else:
    print("\nLogistic Regression performed better on this dataset.")


========== MODEL COMPARISON ==========
                 Model   ROC_AUC
0  Logistic Regression  0.805611
1        Random Forest  0.881383

Random Forest performed better on this dataset.


In [ ]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Precision": [
        precision_score(y_test, lr_predictions),
        precision_score(y_test, rf_predictions)
    ],
    "Recall": [
        recall_score(y_test, lr_predictions),
        recall_score(y_test, rf_predictions)
    ],
    "F1 Score": [
        f1_score(y_test, lr_predictions),
        f1_score(y_test, rf_predictions)
    ],
    "ROC_AUC": [
        lr_auc,
        rf_auc
    ]
})
print(comparison)

                 Model  Precision    Recall  F1 Score   ROC_AUC
0  Logistic Regression   0.062701  0.661017  0.114537  0.805611
1        Random Forest   0.097852  0.694915  0.171548  0.881383
